# Notebook 03: Out-of-Sample Prognoziranje Volatilnosti

**Projekt:** Financijske mreže na ZSE (CROBEX, 2004–2026)
**Teme:** Out-of-sample evaluacija, GARCH(1,1), OLS expanding window, QLIKE gubitak, DM test
**Kolegiji:** Strojno učenje u financijama, Financijsko modeliranje

## Kontekst

Uspoređujemo prognoznu točnost triju modela za **buduću volatilnost** CROBEX tržišta:

- **AR(1)** — autoregresija na sekvenci volatilnosti (standardni baseline)
- **NET** — OLS gdje je prediktor *mrežna mjera* (bez prošle volatilnosti)
- **GARCH(1,1)** — standardna referentna točka u praksi (ako arch paket dostupan)

**Ključno:** expanding window — model se uvijek trenira samo na prošlim podacima.
Kršenje ovog pravila (look-ahead bias) je najčešća greška u ML za financije.


In [ ]:
# Postavljanje okolisa (automatski detektira Google Colab)
import sys
if "google.colab" in sys.modules:
    import subprocess, os
    subprocess.run(["git", "clone", "--branch", "teaching",
                    "https://github.com/svlah-sketch/FinNet.git", "/content/FinNet"], check=True)
    os.chdir("/content/FinNet/teaching/notebooks")
    print("Colab: repozitorij kloniran.")
else:
    print("Lokalno pokretanje — nema potrebe za klonom.")

In [ ]:
import warnings
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

DATA_DIR = Path("../sample_data")

# CROBEX dnevni log-povrati
crobex_raw = pd.read_csv(DATA_DIR / "CROBEX_values.csv", sep=";", encoding="cp1250")
crobex_raw["date"]   = pd.to_datetime(crobex_raw["date"], dayfirst=True)
crobex_raw["crobex"] = pd.to_numeric(crobex_raw["crobex"], errors="coerce")
crobex_ret = np.log(crobex_raw.sort_values("date").set_index("date")["crobex"]).diff().dropna()

# Revizijski prozori i mrežne mjere
revs = pd.read_csv(DATA_DIR / "Revisions.csv", sep=";", encoding="cp1250")
revs["start_date"] = pd.to_datetime(revs["start_date"], dayfirst=True)
revs["end_date"]   = pd.to_datetime(revs["end_date"],   dayfirst=True)
revs = revs[(revs["rev_cnt"] >= 10) & (revs["rev_cnt"] <= 52)].reset_index(drop=True)

metrics = pd.read_csv(DATA_DIR / "sample_metrics_revision.csv",
                      index_col="window_end", parse_dates=True)

print(f"CROBEX: {len(crobex_ret)} dnevnih povrata")
print(f"Revizijskih prozora: {len(revs)}")

## 1. Realizirana volatilnost po prozoru

Za svaki revizijski prozor izračunavamo anualiziranu standardnu devijaciju:

$$\sigma_t = \text{std}(r_d, d \in [t_{start}, t_{end}]) \times \sqrt{252}$$

Cilj prognoze je $\sigma_{t+1}$ — volatilnost *sljedećeg* prozora.


In [ ]:
rows = []
for i, row in revs.iterrows():
    r = crobex_ret.loc[row["start_date"]:row["end_date"]].dropna()
    vol = float(r.std(ddof=1) * np.sqrt(252)) if len(r) >= 10 else np.nan
    diffs = (metrics.index - row["end_date"]).to_series().abs()
    best  = int(diffs.values.argmin())
    met_row = metrics.iloc[best] if diffs.iloc[best].days <= 5 else pd.Series(dtype=float)
    entry = {"end_date": row["end_date"], "start_date": row["start_date"], "vol": vol}
    for col in metrics.columns:
        entry[col] = float(met_row[col]) if col in met_row.index else np.nan
    rows.append(entry)

df = pd.DataFrame(rows)
df["vol_next"] = df["vol"].shift(-1)
df = df.iloc[:-1].copy()
print(f"Prozora za evaluaciju: {df['vol_next'].notna().sum()}")
print(df[["end_date","vol","vol_next","M1_LCC","M3_MeanDeg"]].head(6).to_string(index=False))

## 2. OLS modeli — expanding window

Za svaki prozor $t$ (počevši od `MIN_TRAIN`):
- Treniraj OLS samo na prozorima $1, 2, ..., t-1$
- Prognozi $\hat{\sigma}_{t+1}$ koristeći vrijednosti u trenutku $t$


In [ ]:
MIN_TRAIN = 15

def ols_predict(y_train, X_train, x_pred):
    try:
        X_c = np.column_stack([np.ones(len(X_train)), X_train])
        coef, *_ = np.linalg.lstsq(X_c, y_train, rcond=None)
        return float(coef @ np.concatenate([[1.0], x_pred]))
    except Exception:
        return np.nan

N = len(df)
ar1_hat = np.full(N, np.nan)
net_hat = {col: np.full(N, np.nan) for col in metrics.columns}

for t in range(MIN_TRAIN, N):
    y   = df["vol_next"].iloc[:t].values
    lag = df["vol"].iloc[:t].values
    ok  = ~np.isnan(y) & ~np.isnan(lag)
    if ok.sum() >= 5:
        p = ols_predict(y[ok], lag[ok].reshape(-1,1), [df["vol"].iloc[t]])
        ar1_hat[t] = max(p, 0.01) if not np.isnan(p) else np.nan
    for col in metrics.columns:
        met = df[col].iloc[:t].values
        ok_m = ~np.isnan(y) & ~np.isnan(met)
        if ok_m.sum() >= 5:
            p = ols_predict(y[ok_m], met[ok_m].reshape(-1,1), [df[col].iloc[t]])
            net_hat[col][t] = max(p, 0.01) if not np.isnan(p) else np.nan

df["vol_hat_ar1"] = ar1_hat
for col in metrics.columns:
    df[f"net_{col}"] = net_hat[col]
print("Modeli fitani.")

## 3. QLIKE gubitak i Diebold-Mariano test

**QLIKE gubitak** (standardan za procjenu volatilnosti):
$$\text{QLIKE} = \log(\hat{\sigma}^2) + \frac{\sigma^2_{real}}{\hat{\sigma}^2}$$

**DM test** ($H_0$: jednaka točnost). Negativni DM statistik → NET model bolji od AR(1).


In [ ]:
def qlike(hat, real):
    h2 = hat**2
    return np.log(h2) + real**2 / h2

def dm_test(loss_alt, loss_bench):
    d = loss_alt - loss_bench
    T = len(d)
    dm = d.mean() / np.sqrt(np.var(d, ddof=1) / T)
    k  = (T + 1 - 2 + 1.0/T) / T
    dm_adj = dm * np.sqrt(k)
    p = 2 * (1 - stats.norm.cdf(abs(dm_adj)))
    return float(dm_adj), float(p)

y_r = df["vol_next"].values
ql_ar1 = np.where(~np.isnan(df["vol_hat_ar1"]), qlike(df["vol_hat_ar1"].values, y_r), np.nan)

print(f"{'Mjera':<20} {'Mean QL NET':>12} {'Mean QL AR1':>12} {'DM stat':>9} {'p':>8} {'Bolji?':>7}")
print("-" * 72)
for col in metrics.columns:
    hat = df[f"net_{col}"].values
    ql = np.where(~np.isnan(hat), qlike(hat, y_r), np.nan)
    mask = ~np.isnan(ql) & ~np.isnan(ql_ar1)
    if mask.sum() < 5:
        continue
    stat, p = dm_test(ql[mask], ql_ar1[mask])
    bolji = "DA *" if stat < 0 and p < 0.10 else ("-" if p >= 0.10 else "LOŠIJI")
    print(f"{col:<20} {np.nanmean(ql[mask]):>12.4f} {np.nanmean(ql_ar1[mask]):>12.4f} "
          f"{stat:>9.3f} {p:>8.4f} {bolji:>7}")

## 4. Vizualizacija

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(df["end_date"], df["vol_next"], "k-", lw=1.5, label="Realizirana vol.")
ax.plot(df["end_date"], df["vol_hat_ar1"], "b--", lw=1, alpha=0.8, label="AR(1)")
ax.set_title("Realizirana vs. prognozirana volatilnost")
ax.set_ylabel("Anualizirana vol.")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
best_col = "M4_NCom"
mask = df[best_col].notna() & df["vol_next"].notna()
sc = ax.scatter(df.loc[mask, best_col], df.loc[mask, "vol_next"],
                c=df.loc[mask, "end_date"].astype(int), cmap="plasma", s=50, alpha=0.8)
rho = df.loc[mask, [best_col, "vol_next"]].corr(method="spearman").iloc[0, 1]
ax.set_xlabel(f"{best_col} (samo stresni prozori)")
ax.set_ylabel("Sljedeća vol.")
ax.set_title(f"{best_col} vs. buduća vol. (Spearman rho={rho:.3f})")
ax.grid(alpha=0.3)
plt.colorbar(sc, ax=ax, label="Godina")

plt.tight_layout()
plt.savefig("notebook03_figure.png", dpi=120, bbox_inches="tight")
plt.show()

---

## Zadaci za studente

**1.** Zamijenite `M4_NCom` s `M1_LCC` u scatter plotu. Mijenja li se Spearman ρ?

**2.** Implementirajte **kombinirani model AR1NET**: $\hat{\sigma} = \alpha + \beta_1 \sigma_t + \beta_2 m_t$.
   Usporedite DM test s AR(1) baseline-om.

**3.** Smanjite `MIN_TRAIN` s 15 na 8 prozora. Što se događa s brojem OOS prozora?
   Poboljšavaju li se ili pogoršavaju procjene?

**4.** (Napredni) Zamijenite QLIKE s MSE gubidtkom: $(\hat{\sigma} - \sigma_{real})^2$.
   Mijenja li se poredak modela? QLIKE penalizira podcjenjivanje volatilnosti više od MSE — zašto je to važno u upravljanju rizikom?
